# 13 Registro afin por mascaras - todos los sujetos

Este notebook prueba el siguiente paso despues del registro rigido: **registro afin H&E -> HSI basado en mascaras**.

La idea es:

1. Cargar las imagenes y mascaras ya guardadas en `Imagenes_a_escala`.
2. Calcular el baseline rigido igual que en el notebook 12.
3. Usar ese rigido como punto de partida.
4. Refinar con una transformacion afin usando mapas de distancia de las mascaras.
5. Comparar rigido vs afin con Dice, IoU y distancia entre contornos.

Importante: el afin se acepta solo si mejora el rigid y si la matriz no introduce escalas/cizallas exageradas.


In [ ]:
from pathlib import Path
import csv
import json
import math

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


BASE_DIR = Path(r'Registration\DeepLearning')
INPUT_DIR = BASE_DIR / 'Imagenes_a_escala'
OUTPUT_ROOT = BASE_DIR / 'Tecnicas_registration' / '00_baseline_clasico' / 'outputs' / 'outputs_registro_afin_mascaras'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUBJECTS = ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']

print('Entrada:', INPUT_DIR)
print('Salida:', OUTPUT_ROOT)


## Funciones base

Estas funciones son casi las mismas del registro rigido: leer imagenes, leer mascaras, calcular centroides, aplicar matrices y medir solapamiento.


In [ ]:
def load_rgb(path):
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)


def load_mask(path):
    return np.asarray(Image.open(path).convert('L'), dtype=np.uint8) > 0


def subject_paths(subject_id):
    subject_dir = INPUT_DIR / subject_id
    return {
        'he_rgb': subject_dir / f'{subject_id}_h&e.png',
        'hsi_rgb': subject_dir / f'{subject_id}_hsi.png',
        'he_mask': subject_dir / f'{subject_id}_he_mask.png',
        'hsi_mask': subject_dir / f'{subject_id}_hsi_mask.png',
        'metadata': subject_dir / f'{subject_id}_metadata.json',
    }


def centroid_xy(mask):
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        raise ValueError('Mascara vacia: no se puede calcular centroide')
    return np.array([xs.mean(), ys.mean()], dtype=np.float32)


def rigid_matrix_moving_to_fixed(moving_mask, fixed_mask, angle_deg):
    moving_centroid = centroid_xy(moving_mask)
    fixed_centroid = centroid_xy(fixed_mask)
    matrix = cv2.getRotationMatrix2D(tuple(moving_centroid), float(angle_deg), 1.0)
    rotated_centroid = matrix[:, :2] @ moving_centroid + matrix[:, 2]
    matrix[:, 2] += fixed_centroid - rotated_centroid
    return matrix.astype(np.float32)


def to_homogeneous(matrix_2x3):
    matrix_3x3 = np.eye(3, dtype=np.float32)
    matrix_3x3[:2, :] = np.asarray(matrix_2x3, dtype=np.float32)
    return matrix_3x3


def from_homogeneous(matrix_3x3):
    return np.asarray(matrix_3x3[:2, :], dtype=np.float32)


def compose_affines(left_after_right, right_first):
    return from_homogeneous(to_homogeneous(left_after_right) @ to_homogeneous(right_first))


def warp_to_fixed_canvas(image, fixed_shape, matrix, is_mask=False):
    fixed_h, fixed_w = fixed_shape[:2]
    if is_mask:
        warped = cv2.warpAffine(
            (np.asarray(image) > 0).astype(np.uint8) * 255,
            matrix,
            (fixed_w, fixed_h),
            flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0,
        )
        return warped > 0

    return cv2.warpAffine(
        np.asarray(image, dtype=np.uint8),
        matrix,
        (fixed_w, fixed_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )


def dice_score(mask_a, mask_b):
    a = np.asarray(mask_a) > 0
    b = np.asarray(mask_b) > 0
    denom = a.sum() + b.sum()
    if denom == 0:
        return 1.0
    return float(2.0 * np.logical_and(a, b).sum() / denom)


def iou_score(mask_a, mask_b):
    a = np.asarray(mask_a) > 0
    b = np.asarray(mask_b) > 0
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(a, b).sum() / union)


def contour_mask(mask, thickness=1):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    out = np.zeros_like(mask_u8)
    cv2.drawContours(out, contours, -1, 255, thickness=int(thickness))
    return out > 0


def symmetric_contour_distance(mask_a, mask_b):
    contour_a = contour_mask(mask_a)
    contour_b = contour_mask(mask_b)
    if not contour_a.any() or not contour_b.any():
        return {'mean': float('nan'), 'p95': float('nan')}

    dist_to_b = cv2.distanceTransform((~contour_b).astype(np.uint8), cv2.DIST_L2, 5)
    dist_to_a = cv2.distanceTransform((~contour_a).astype(np.uint8), cv2.DIST_L2, 5)
    distances = np.concatenate([dist_to_b[contour_a], dist_to_a[contour_b]])
    return {
        'mean': float(np.mean(distances)),
        'p95': float(np.percentile(distances, 95)),
    }


def save_mask(path, mask):
    Image.fromarray((np.asarray(mask) > 0).astype(np.uint8) * 255).save(path)


def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, '') for key in fieldnames})


## Visualizacion

Rojo = HSI fija. Verde/cian = H&E movida despues de aplicar la transformacion.


In [ ]:
def normalize_u8(gray):
    arr = np.asarray(gray, dtype=np.float32)
    lo, hi = np.percentile(arr, [1, 99])
    if hi <= lo:
        return np.zeros(arr.shape, dtype=np.uint8)
    return (np.clip((arr - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)


def overlay_rgb_images(fixed_rgb, moving_rgb, fixed_mask=None, moving_mask=None):
    fixed_gray = cv2.cvtColor(np.asarray(fixed_rgb, dtype=np.uint8), cv2.COLOR_RGB2GRAY)
    moving_gray = cv2.cvtColor(np.asarray(moving_rgb, dtype=np.uint8), cv2.COLOR_RGB2GRAY)
    fixed_norm = normalize_u8(fixed_gray)
    moving_norm = normalize_u8(moving_gray)

    overlay = np.zeros((*fixed_gray.shape, 3), dtype=np.uint8)
    overlay[..., 0] = fixed_norm
    overlay[..., 1] = moving_norm
    overlay[..., 2] = fixed_norm // 3

    if fixed_mask is not None:
        overlay[contour_mask(fixed_mask, thickness=2)] = np.array([255, 0, 0], dtype=np.uint8)
    if moving_mask is not None:
        overlay[contour_mask(moving_mask, thickness=2)] = np.array([0, 255, 255], dtype=np.uint8)
    return overlay


def contours_overlay_image(fixed_mask, moving_mask):
    out = np.zeros((*fixed_mask.shape, 3), dtype=np.uint8)
    out[contour_mask(fixed_mask, thickness=2)] = np.array([255, 0, 0], dtype=np.uint8)
    out[contour_mask(moving_mask, thickness=2)] = np.array([0, 255, 255], dtype=np.uint8)
    return out


## Busqueda rigida inicial

Antes de afinar con una matriz afin, repetimos la busqueda de angulo. Esto nos da una inicializacion fuerte y una comparacion justa.


In [ ]:
def evaluate_angle(he_mask, hsi_mask, angle_deg):
    matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, angle_deg)
    warped_mask = warp_to_fixed_canvas(he_mask, hsi_mask.shape, matrix, is_mask=True)
    return {
        'angle_deg': float(angle_deg),
        'dice': dice_score(warped_mask, hsi_mask),
        'iou': iou_score(warped_mask, hsi_mask),
    }


def search_best_angle(he_mask, hsi_mask, coarse_step=2.0, fine_radius=3.0, fine_step=0.25):
    coarse_angles = np.arange(-180.0, 180.0 + 1e-6, coarse_step)
    coarse_results = [evaluate_angle(he_mask, hsi_mask, angle) for angle in coarse_angles]
    best_coarse = max(coarse_results, key=lambda row: row['dice'])

    fine_angles = np.arange(
        best_coarse['angle_deg'] - fine_radius,
        best_coarse['angle_deg'] + fine_radius + 1e-6,
        fine_step,
    )
    fine_results = [evaluate_angle(he_mask, hsi_mask, angle) for angle in fine_angles]
    best_fine = max(fine_results, key=lambda row: row['dice'])
    return best_fine, coarse_results + fine_results


## Refinamiento afin

Para refinar no usamos directamente RGB, porque H&E y HSI son modalidades distintas. Convertimos cada mascara en un **mapa de distancia firmado**:

- positivo dentro del especimen,
- negativo fuera,
- suave cerca del contorno.

Esto permite que ECC tenga una imagen continua sobre la que optimizar, en vez de una mascara binaria brusca.


In [ ]:
def signed_distance_field(mask, clip_percentile=98):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
    inside = cv2.distanceTransform(mask_u8, cv2.DIST_L2, 5)
    outside = cv2.distanceTransform(1 - mask_u8, cv2.DIST_L2, 5)
    signed = inside - outside
    clip = float(np.percentile(np.abs(signed), clip_percentile))
    if not np.isfinite(clip) or clip <= 0:
        clip = 1.0
    signed = np.clip(signed, -clip, clip)
    return ((signed + clip) / (2.0 * clip)).astype(np.float32)


def affine_matrix_diagnostics(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    linear = matrix[:, :2]
    det = float(np.linalg.det(linear))
    try:
        singular_values = np.linalg.svd(linear, compute_uv=False)
    except np.linalg.LinAlgError:
        singular_values = np.array([np.nan, np.nan], dtype=np.float32)

    col0 = linear[:, 0]
    col1 = linear[:, 1]
    norm0 = float(np.linalg.norm(col0))
    norm1 = float(np.linalg.norm(col1))
    cos_angle = float(np.dot(col0, col1) / max(norm0 * norm1, 1e-6))

    return {
        'determinant': det,
        'scale_min_svd': float(np.nanmin(singular_values)),
        'scale_max_svd': float(np.nanmax(singular_values)),
        'axis_cosine': cos_angle,
    }


def is_reasonable_affine(matrix, scale_min=0.70, scale_max=1.30, max_axis_cosine=0.45):
    diag = affine_matrix_diagnostics(matrix)
    if not np.isfinite(diag['determinant']) or diag['determinant'] <= 0:
        return False, diag, 'determinant_non_positive'
    if diag['scale_min_svd'] < scale_min or diag['scale_max_svd'] > scale_max:
        return False, diag, 'scale_out_of_range'
    if abs(diag['axis_cosine']) > max_axis_cosine:
        return False, diag, 'excessive_shear'
    return True, diag, 'ok'


def estimate_affine_after_rigid(he_mask, hsi_mask, rigid_matrix):
    he_mask_rigid = warp_to_fixed_canvas(he_mask, hsi_mask.shape, rigid_matrix, is_mask=True)
    fixed_field = signed_distance_field(hsi_mask)
    moving_field = signed_distance_field(he_mask_rigid)

    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 300, 1e-6)
    delta_init = np.eye(2, 3, dtype=np.float32)

    try:
        cc, delta = cv2.findTransformECC(
            fixed_field,
            moving_field,
            delta_init,
            cv2.MOTION_AFFINE,
            criteria,
            None,
            5,
        )
    except cv2.error as exc:
        return {
            'success': False,
            'message': str(exc),
            'delta_matrix': np.eye(2, 3, dtype=np.float32),
            'total_matrix': rigid_matrix,
            'candidate_mask': he_mask_rigid,
            'candidate_dice': dice_score(he_mask_rigid, hsi_mask),
            'candidate_iou': iou_score(he_mask_rigid, hsi_mask),
            'ecc_cc': float('nan'),
            'delta_used': 'identity_after_failure',
        }

    # OpenCV ECC tiene una convencion de matriz que depende de como se aplique warpAffine.
    # Probamos delta directa e inversa y nos quedamos con la que mejora mas la mascara.
    delta_inv = cv2.invertAffineTransform(delta)
    candidates = []
    for name, delta_candidate in [('direct', delta), ('inverse', delta_inv)]:
        total = compose_affines(delta_candidate, rigid_matrix)
        mask_candidate = warp_to_fixed_canvas(he_mask, hsi_mask.shape, total, is_mask=True)
        candidates.append({
            'name': name,
            'delta': delta_candidate.astype(np.float32),
            'total': total.astype(np.float32),
            'mask': mask_candidate,
            'dice': dice_score(mask_candidate, hsi_mask),
            'iou': iou_score(mask_candidate, hsi_mask),
        })

    best = max(candidates, key=lambda row: row['dice'])
    return {
        'success': True,
        'message': 'ok',
        'delta_matrix': best['delta'],
        'total_matrix': best['total'],
        'candidate_mask': best['mask'],
        'candidate_dice': best['dice'],
        'candidate_iou': best['iou'],
        'ecc_cc': float(cc),
        'delta_used': best['name'],
    }


## Procesamiento por sujeto

El criterio de aceptacion es deliberadamente conservador:

- si el afin mejora Dice frente al rigid, se puede aceptar;
- si la matriz tiene escalas o shear exagerados, se rechaza;
- si ECC falla, se deja el rigid como resultado final y se registra el fallo.


In [ ]:
def process_subject(subject_id):
    paths = subject_paths(subject_id)
    he_rgb = load_rgb(paths['he_rgb'])
    hsi_rgb = load_rgb(paths['hsi_rgb'])
    he_mask = load_mask(paths['he_mask'])
    hsi_mask = load_mask(paths['hsi_mask'])
    metadata = json.loads(paths['metadata'].read_text(encoding='utf-8'))

    out_dir = OUTPUT_ROOT / subject_id
    out_dir.mkdir(parents=True, exist_ok=True)

    best_rigid, angle_results = search_best_angle(he_mask, hsi_mask)
    rigid_matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, best_rigid['angle_deg'])
    he_mask_rigid = warp_to_fixed_canvas(he_mask, hsi_mask.shape, rigid_matrix, is_mask=True)
    he_rgb_rigid = warp_to_fixed_canvas(he_rgb, hsi_mask.shape, rigid_matrix, is_mask=False)
    rigid_contour_distance = symmetric_contour_distance(he_mask_rigid, hsi_mask)

    affine_result = estimate_affine_after_rigid(he_mask, hsi_mask, rigid_matrix)
    affine_reasonable, affine_diag, affine_reason = is_reasonable_affine(affine_result['total_matrix'])
    improves_rigid = affine_result['candidate_dice'] >= best_rigid['dice'] + 1e-4
    affine_accepted = bool(affine_result['success'] and affine_reasonable and improves_rigid)
    if not affine_result['success']:
        final_note = 'affine_rejected_ecc_failed'
    elif not improves_rigid:
        final_note = 'affine_rejected_no_improvement'
    elif not affine_reasonable:
        final_note = f'affine_rejected_{affine_reason}'
    else:
        final_note = 'affine_accepted'

    if affine_accepted:
        final_matrix = affine_result['total_matrix']
        he_mask_affine = affine_result['candidate_mask']
    else:
        final_matrix = rigid_matrix
        he_mask_affine = he_mask_rigid

    he_rgb_affine = warp_to_fixed_canvas(he_rgb, hsi_mask.shape, final_matrix, is_mask=False)
    affine_contour_distance = symmetric_contour_distance(he_mask_affine, hsi_mask)

    overlay = overlay_rgb_images(hsi_rgb, he_rgb_affine, hsi_mask, he_mask_affine)
    contour_overlay = contours_overlay_image(hsi_mask, he_mask_affine)

    files = {
        'he_rgb_affine': out_dir / f'{subject_id}_he_affine_to_hsi.png',
        'he_mask_affine': out_dir / f'{subject_id}_he_mask_affine_to_hsi.png',
        'overlay_rgb': out_dir / f'{subject_id}_overlay_rgb_affine.png',
        'overlay_contours': out_dir / f'{subject_id}_overlay_contours_affine.png',
        'metrics': out_dir / f'{subject_id}_affine_metrics.json',
        'angle_search': out_dir / f'{subject_id}_angle_search.csv',
    }

    Image.fromarray(he_rgb_affine).save(files['he_rgb_affine'])
    save_mask(files['he_mask_affine'], he_mask_affine)
    Image.fromarray(overlay).save(files['overlay_rgb'])
    Image.fromarray(contour_overlay).save(files['overlay_contours'])
    write_csv(files['angle_search'], angle_results, ['angle_deg', 'dice', 'iou'])

    metrics = {
        'subject_id': subject_id,
        'registration_direction': 'he_to_hsi',
        'fixed_image': 'hsi',
        'moving_image': 'he',
        'best_rigid_angle_deg': float(best_rigid['angle_deg']),
        'rigid_matrix_2x3': rigid_matrix.tolist(),
        'affine_total_matrix_2x3': np.asarray(final_matrix).tolist(),
        'affine_delta_matrix_2x3': np.asarray(affine_result['delta_matrix']).tolist(),
        'affine_delta_used': affine_result['delta_used'],
        'affine_ecc_success': bool(affine_result['success']),
        'affine_ecc_message': affine_result['message'],
        'affine_ecc_cc': affine_result['ecc_cc'],
        'affine_accepted': affine_accepted,
        'affine_final_note': final_note,
        'affine_reject_reason': affine_reason,
        'affine_matrix_determinant': affine_diag['determinant'],
        'affine_scale_min_svd': affine_diag['scale_min_svd'],
        'affine_scale_max_svd': affine_diag['scale_max_svd'],
        'affine_axis_cosine': affine_diag['axis_cosine'],
        'dice_rigid': dice_score(he_mask_rigid, hsi_mask),
        'dice_affine_candidate': affine_result['candidate_dice'],
        'dice_affine_final': dice_score(he_mask_affine, hsi_mask),
        'iou_rigid': iou_score(he_mask_rigid, hsi_mask),
        'iou_affine_candidate': affine_result['candidate_iou'],
        'iou_affine_final': iou_score(he_mask_affine, hsi_mask),
        'contour_mean_distance_rigid_px': rigid_contour_distance['mean'],
        'contour_p95_distance_rigid_px': rigid_contour_distance['p95'],
        'contour_mean_distance_affine_px': affine_contour_distance['mean'],
        'contour_p95_distance_affine_px': affine_contour_distance['p95'],
        'target_px_per_cm': metadata.get('target_px_per_cm'),
        'overlay_rgb_path': str(files['overlay_rgb']),
        'overlay_contours_path': str(files['overlay_contours']),
    }

    files['metrics'].write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    return metrics, overlay, contour_overlay


## Ejecutar todos los sujetos

In [ ]:
all_metrics = []
overlay_images = {}
contour_images = {}

for subject_id in SUBJECTS:
    print(f'=== {subject_id} ===')
    metrics, overlay, contour_overlay = process_subject(subject_id)
    all_metrics.append(metrics)
    overlay_images[subject_id] = overlay
    contour_images[subject_id] = contour_overlay
    print(
        f"rigid Dice={metrics['dice_rigid']:.4f} -> "
        f"affine final Dice={metrics['dice_affine_final']:.4f} | "
        f"accepted={metrics['affine_accepted']} | "
        f"reason={metrics['affine_final_note']}"
    )

summary_csv = OUTPUT_ROOT / 'affine_mask_summary.csv'
summary_json = OUTPUT_ROOT / 'affine_mask_summary.json'
summary_fields = [
    'subject_id',
    'best_rigid_angle_deg',
    'affine_accepted',
    'affine_final_note',
    'dice_rigid',
    'dice_affine_candidate',
    'dice_affine_final',
    'iou_rigid',
    'iou_affine_candidate',
    'iou_affine_final',
    'contour_mean_distance_rigid_px',
    'contour_mean_distance_affine_px',
    'contour_p95_distance_rigid_px',
    'contour_p95_distance_affine_px',
    'affine_matrix_determinant',
    'affine_scale_min_svd',
    'affine_scale_max_svd',
    'affine_axis_cosine',
    'target_px_per_cm',
    'overlay_rgb_path',
    'overlay_contours_path',
]
write_csv(summary_csv, all_metrics, summary_fields)
summary_json.write_text(json.dumps(all_metrics, indent=2), encoding='utf-8')

print('Resumen CSV:', summary_csv)
print('Resumen JSON:', summary_json)


## Resumen numerico

In [ ]:
for row in all_metrics:
    delta_dice = row['dice_affine_final'] - row['dice_rigid']
    delta_dist = row['contour_mean_distance_affine_px'] - row['contour_mean_distance_rigid_px']
    print(
        f"{row['subject_id']}: "
        f"Dice rigid={row['dice_rigid']:.4f}, affine={row['dice_affine_final']:.4f} "
        f"(delta {delta_dice:+.4f}) | "
        f"contour_mean rigid={row['contour_mean_distance_rigid_px']:.2f}px, "
        f"affine={row['contour_mean_distance_affine_px']:.2f}px "
        f"(delta {delta_dist:+.2f}px) | "
        f"accepted={row['affine_accepted']}"
    )


## Montajes visuales

In [ ]:
def resize_for_montage(img, target_width=360):
    h, w = img.shape[:2]
    scale = target_width / max(w, 1)
    target_h = max(1, int(round(h * scale)))
    return cv2.resize(img, (target_width, target_h), interpolation=cv2.INTER_AREA)


def add_label(img, label):
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 28), (0, 0, 0), -1)
    cv2.putText(out, label, (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def make_grid(images, labels, cols=3, target_width=360):
    resized = [add_label(resize_for_montage(img, target_width), label) for img, label in zip(images, labels)]
    rows = int(math.ceil(len(resized) / cols))
    max_h = max(img.shape[0] for img in resized)
    cell_w = target_width
    canvas = np.zeros((rows * max_h, cols * cell_w, 3), dtype=np.uint8)
    for idx, img in enumerate(resized):
        r = idx // cols
        c = idx % cols
        y = r * max_h
        x = c * cell_w
        canvas[y:y + img.shape[0], x:x + img.shape[1]] = img
    return canvas


labels = [
    f"{row['subject_id']} | Dice {row['dice_affine_final']:.3f} | {'afin' if row['affine_accepted'] else 'rigid'}"
    for row in all_metrics
]
overlay_grid = make_grid([overlay_images[sid] for sid in SUBJECTS], labels)
contour_grid = make_grid([contour_images[sid] for sid in SUBJECTS], labels)

overlay_grid_path = OUTPUT_ROOT / 'affine_mask_overlay_grid.png'
contour_grid_path = OUTPUT_ROOT / 'affine_mask_contour_grid.png'
Image.fromarray(overlay_grid).save(overlay_grid_path)
Image.fromarray(contour_grid).save(contour_grid_path)

print('Montaje overlays:', overlay_grid_path)
print('Montaje contornos:', contour_grid_path)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
axes[0].imshow(overlay_grid)
axes[0].axis('off')
axes[0].set_title('Registro afin por mascaras - overlays')
axes[1].imshow(contour_grid)
axes[1].axis('off')
axes[1].set_title('Registro afin por mascaras - contornos')
plt.show()


## Como leer estos resultados

- `dice_affine_final` se compara contra `dice_rigid`.
- Si `affine_accepted=False`, el resultado final queda igual que el rigid porque el refinamiento no era fiable.
- Si mejora poco o se rechaza casi siempre, significa que el problema principal no es afin sino no rigido.
- Si mejora bastante en algun sujeto, esa transformacion afin deberia ser el punto de partida del siguiente paso deformable.
